In [ ]:
### Real Estate Project
import pandas as pd
import numpy as np
import matplotlib as plt 


In [ ]:
# scrapping data
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

base_url = "https://nigeriapropertycentre.com/for-sale/houses/oyo/ibadan?page={}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

properties = []

for page in range(1, 51):  # scrape 10 pages (≈ 200 listings)
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    listings = soup.find_all("div", class_="wp-block property list")

    print(f"Page {page}: {len(listings)} listings found")

    for listing in listings:
        try:
            title = listing.find("h4").text.strip()
            location = listing.find("address").text.strip()

            prices = listing.find_all("span", class_="price")
            price = prices[1].text.strip() if len(prices) > 1 else prices[0].text.strip()

            # Extract features
            features = listing.find_all("li")

            bedrooms = bathrooms = toilets = None

            for f in features:
                text = f.text.lower()

                # Bedrooms
                if "bed" in text:
                    bedrooms = int(''.join(filter(str.isdigit, text)))

                # Bathrooms
                elif "bath" in text:
                    bathrooms = int(''.join(filter(str.isdigit, text)))

                # Toilets
                elif "toilet" in text:
                    toilets = int(''.join(filter(str.isdigit, text)))

                # Size (sqm, sqft, plots, acres)
                elif any(unit in text for unit in ["sqm", "sq m", "square meter", "sqft", "plot", "acre"]):
                    size = text.strip()

            properties.append({
                "title": title,
                "price": price,
                "location": location,
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "toilets": toilets,
                "size": size
            })

        except:
            continue
        
    time.sleep(2)  # to avoid blocking

df = pd.DataFrame(properties)

df.to_csv("ibadan_properties.csv", index=False)

print(df.info())

Page 1: 20 listings found
Page 2: 20 listings found
Page 3: 20 listings found
Page 4: 20 listings found
Page 5: 20 listings found
Page 6: 20 listings found
Page 7: 20 listings found
Page 8: 20 listings found
Page 9: 20 listings found
Page 10: 20 listings found
Page 11: 20 listings found
Page 12: 20 listings found
Page 13: 20 listings found
Page 14: 20 listings found
Page 15: 20 listings found
Page 16: 20 listings found
Page 17: 20 listings found
Page 18: 20 listings found
Page 19: 20 listings found
Page 20: 20 listings found
Page 21: 20 listings found
Page 22: 20 listings found
Page 23: 20 listings found
Page 24: 20 listings found
Page 25: 20 listings found
Page 26: 20 listings found
Page 27: 20 listings found
Page 28: 20 listings found
Page 29: 20 listings found
Page 30: 20 listings found
Page 31: 20 listings found
Page 32: 20 listings found
Page 33: 20 listings found
Page 34: 20 listings found
Page 35: 20 listings found
Page 36: 20 listings found
Page 37: 20 listings found
Page 38: 2

In [69]:
# extracting size value
def extract_size_value(size):
    if size:
        match = re.search(r'\d+', size)
        return int(match.group()) if match else None
    return None

# Extracting property type
def extract_property_type(title):
    types = ['duplex', 'flat', 'bungalow', 'apartment', 'land']
    for t in types:
        if t in title.lower():
            return t
    return 'other'

df['size_value'] = df['size'].apply(extract_size_value)
df['property_type'] = df['title'].apply(extract_property_type)


print(df.head())
print(df.tail())

                                  title        price  \
0  3 bedroom detached bungalow for sale   99,750,000   
1  9 bedroom detached bungalow for sale   55,000,000   
2  4 bedroom detached bungalow for sale   50,000,000   
3     2 bedroom block of flats for sale  130,000,000   
4    4 bedroom detached duplex for sale  340,000,000   

                                            location  bedrooms  bathrooms  \
0  Adron City Park And Gardens, Asejire, Ibadan, Oyo       3.0        4.0   
1      Close To Makoni Hotel, Ologuneru, Ibadan, Oyo       9.0        NaN   
2  Road 4 Ire-akari, Idi-oya, Ayegun Oleyo Road, ...       4.0        NaN   
3           Orange Gate, Oluyole Estate, Ibadan, Oyo       2.0        2.0   
4              Blue Gate Oluyole Estate, Ibadan, Oyo       4.0        4.0   

   toilets                  size  size_value property_type  
0      4.0    500 sqm total area         500      bungalow  
1      9.0  605 sqm covered area         605      bungalow  
2      NaN  500 s

In [ ]:
# Data cleaning
df['price'] = df['price'].str.replace(',', '', regex=False)

df['location'] = df['location'].str.split(',').str[:2].str.join(', ')

print(df.head())


                                  title      price  \
0  3 bedroom detached bungalow for sale   99750000   
1  9 bedroom detached bungalow for sale   55000000   
2  4 bedroom detached bungalow for sale   50000000   
3     2 bedroom block of flats for sale  130000000   
4    4 bedroom detached duplex for sale  340000000   

                                location  bedrooms  bathrooms  toilets  \
0  Adron City Park And Gardens,  Asejire       3.0        4.0      4.0   
1      Close To Makoni Hotel,  Ologuneru       9.0        NaN      9.0   
2             Road 4 Ire-akari,  Idi-oya       4.0        NaN      NaN   
3           Orange Gate,  Oluyole Estate       2.0        2.0      2.0   
4      Blue Gate Oluyole Estate,  Ibadan       4.0        4.0      4.0   

                   size  size_value property_type  
0    500 sqm total area         500      bungalow  
1  605 sqm covered area         605      bungalow  
2  500 sqm covered area         500      bungalow  
3  500 sqm covered are

In [72]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 940 entries, 0 to 939
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          940 non-null    object 
 1   price          940 non-null    object 
 2   location       940 non-null    object 
 3   bedrooms       798 non-null    float64
 4   bathrooms      554 non-null    float64
 5   toilets        564 non-null    float64
 6   size           940 non-null    object 
 7   size_value     940 non-null    int64  
 8   property_type  940 non-null    object 
dtypes: float64(3), int64(1), object(5)
memory usage: 66.2+ KB
None


In [ ]:
# Data Cleaning

In [ ]:
# Feature engineering

df['price_per_bedroom'] = df['price'] / df['bedrooms']

df['is_luxury'] = df['price'] > 50_000_000

TypeError: unsupported operand type(s) for /: 'str' and 'float'

In [ ]:
# loading data
df = pd.read_csv("ibadan_real_estate.csv")

# inspecting data
# Preview
print(df.head())

# info
print(df.info())

# Check missing value
print(df.isnull().sum())

In [ ]:
# Data Cleaning
# Fill missing numeric values with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)
    
# Remove duplicates
df.drop_duplicates(inplace=True)

In [ ]:
## Exploratory Data Analysis
# Price Distribution
plt.figure()
sns.histplote(df['price'], kde=True)
plt.title("Property Price Distribution")
plt.show()

In [ ]:
# Price by Location
plt.figure()
sns.boxplot(x='location', y='price', data=df)
plt.xticks(rotation=45)
plt.title("Price by Location")
plt.show()

# Correlation Heatmap
plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
np.random.seed(42)

locations = ["Bodija", "Akobo", "Jericho", "Oluyole", "Challenge", "Ring Road"]
property_types = ["Duplex", "Bungalow", "Flat", "Land"]

n = 300

# creating demo dataset
data = pd.DataFrame({
    "location": np.random.choice(locations, n),
    "property_types": np.random.choice(property_types, n),
    "bedrooms": np.random.randint(1, 6, n),
    "bathrooms": np.random.randint(1, 5, n),
    "size_sqm": np.random.randint(50, 500, n),
    "price": np.random.randint(5_000_000, 150_000_000, n),
    "days_on_market": np.random.randint(5, 180, n)
})

data.head()

,location,property_types,bedrooms,bathrooms,size_sqm,price,days_on_market
0,Oluyole,Land,2,1,369,96636655,92
1,Challenge,Bungalow,1,1,398,109904657,113
2,Jericho,Duplex,4,4,377,60842822,167
3,Challenge,Flat,3,2,316,40393865,74
4,Challenge,Duplex,4,4,410,103820870,162


In [3]:
print(data.info())
print(data.describe())

# Average price per location
print(data.groupby("location")["price"].mean().sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   location        300 non-null    object
 1   property_types  300 non-null    object
 2   bedrooms        300 non-null    int32 
 3   bathrooms       300 non-null    int32 
 4   size_sqm        300 non-null    int32 
 5   price           300 non-null    int32 
 6   days_on_market  300 non-null    int32 
dtypes: int32(5), object(2)
memory usage: 10.7+ KB
None
         bedrooms   bathrooms    size_sqm         price  days_on_market
count  300.000000  300.000000  300.000000  3.000000e+02      300.000000
mean     3.023333    2.470000  281.240000  7.784913e+07       91.036667
std      1.473100    1.095338  128.188859  4.111851e+07       52.451749
min      1.000000    1.000000   50.000000  6.020154e+06        5.000000
25%      2.000000    2.000000  173.500000  4.157984e+07       43.750000
50%      3.0

In [ ]:
df = pd.DataFrame